# Fast Suite Results Overview

Overview for `stage1_model_benchmarks/fast` across horizons 10, 20, and 30.

The notebook loads the `latest` run per horizon, compares point and probabilistic metrics, and shows top-15 feature importance for available models.

In [ ]:
from __future__ import annotations

import io
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'build_price_feature_day.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from build_price_feature_day import make_s3_client

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

BUCKET = 'binance-data-downloader'
RESULTS_PREFIX = 'stage1_model_benchmarks'
SUITE = 'fast'
HORIZONS = [10, 20, 30]

s3 = make_s3_client()

In [ ]:
def read_json(key: str) -> dict:
    return json.loads(s3.get_object(Bucket=BUCKET, Key=key)['Body'].read().decode('utf-8'))


def read_parquet(key: str) -> pd.DataFrame:
    body = s3.get_object(Bucket=BUCKET, Key=key)['Body'].read()
    return pd.read_parquet(io.BytesIO(body))


def object_exists(key: str) -> bool:
    try:
        s3.head_object(Bucket=BUCKET, Key=key)
    except Exception:
        return False
    return True


def latest_base(horizon: int) -> str:
    return f'{RESULTS_PREFIX}/{SUITE}/horizon_{horizon}/latest'


def run_base(horizon: int, run_id: str) -> str:
    return f'{RESULTS_PREFIX}/{SUITE}/horizon_{horizon}/{run_id}'

In [ ]:
configs = []
results = []

for horizon in HORIZONS:
    cfg = read_json(f'{latest_base(horizon)}/run_config.json')
    cfg['horizon'] = horizon
    configs.append(cfg)

    frame = read_parquet(f'{latest_base(horizon)}/stage1_results.parquet')
    frame['horizon'] = horizon
    frame['run_id'] = cfg['run_id']
    results.append(frame)

configs_df = pd.DataFrame(configs)
results_df = pd.concat(results, ignore_index=True)

if 'NRMSE' not in results_df.columns or results_df['NRMSE'].isna().all():
    target_std_by_job = {}
    for cfg in configs:
        horizon = int(cfg['horizon'])
        base = run_base(horizon, cfg['run_id'])
        for job_id in results_df.loc[results_df['horizon'].eq(horizon), 'job_id'].tolist():
            pred_key = f'{base}/jobs/{job_id}/predictions.parquet'
            if not object_exists(pred_key):
                continue
            pred = read_parquet(pred_key)
            target_std_by_job[(horizon, job_id)] = float(pred['y_true'].std(ddof=0))
    results_df['target_std'] = [target_std_by_job.get((h, j), np.nan) for h, j in zip(results_df['horizon'], results_df['job_id'])]
    results_df['NRMSE'] = results_df['RMSE'] / results_df['target_std']
    results_df['NRMSE_std'] = results_df['NRMSE']

configs_df[['horizon', 'run_id', 'dataset_prefix', 'target_column', 'jobs', 'max_parallel_models', 'threads_per_model', 'output_prefix']]

## Metric Summary

In [ ]:
metric_cols = [
    'horizon', 'job_id', 'model_family', 'loss_function',
    'RMSE', 'NRMSE', 'NRMSE_std', 'NRMSE_range', 'MAE', 'Direction_Accuracy', 'Direction_Accuracy_0.25%', 'OOS_R2',
    'PinballLoss_05', 'PinballLoss_50', 'PinballLoss_95',
    'CRPS', 'Coverage90', 'IntervalWidth90', 'NLL',
    'train_time', 'predict_time', 'elapsed_sec',
]

available_metric_cols = [col for col in metric_cols if col in results_df.columns]
ranked = results_df[available_metric_cols].sort_values(['horizon', 'RMSE']).reset_index(drop=True)
ranked

In [ ]:
best_rows = []
for horizon, group in results_df.groupby('horizon'):
    best_rows.append({
        'horizon': horizon,
        'best_RMSE_model': group.loc[group['RMSE'].idxmin(), 'job_id'],
        'best_RMSE': group['RMSE'].min(),
        'best_NRMSE_model': group.loc[group['NRMSE'].idxmin(), 'job_id'],
        'best_NRMSE': group['NRMSE'].min(),
        'best_MAE_model': group.loc[group['MAE'].idxmin(), 'job_id'],
        'best_MAE': group['MAE'].min(),
        'best_DA025_model': group.loc[group['Direction_Accuracy_0.25%'].idxmax(), 'job_id'],
        'best_DA025': group['Direction_Accuracy_0.25%'].max(),
        'best_OOS_R2_model': group.loc[group['OOS_R2'].idxmax(), 'job_id'],
        'best_OOS_R2': group['OOS_R2'].max(),
    })

best_df = pd.DataFrame(best_rows)
best_df

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5), constrained_layout=True)

plot_specs = [
    ('RMSE', True, 'RMSE, lower is better'),
    ('NRMSE', True, 'NRMSE = RMSE / std(y), lower is better'),
    ('MAE', True, 'MAE, lower is better'),
    ('Direction_Accuracy_0.25%', False, 'Direction Accuracy 0.25%, higher is better'),
]

for ax, (metric, ascending, title) in zip(axes, plot_specs):
    pivot = results_df.pivot(index='job_id', columns='horizon', values=metric)
    order = pivot.mean(axis=1).sort_values(ascending=ascending).index
    pivot.loc[order].plot(kind='barh', ax=ax)
    ax.set_title(title)
    ax.set_xlabel(metric)
    ax.set_ylabel('')
    ax.grid(axis='x', alpha=0.25)

plt.show()

## Probabilistic Metrics

In [ ]:
prob_cols = [
    'horizon', 'job_id', 'RMSE', 'NRMSE', 'MAE', 'Direction_Accuracy_0.25%',
    'PinballLoss_05', 'PinballLoss_50', 'PinballLoss_95',
    'CRPS', 'Coverage90', 'IntervalWidth90', 'NLL',
]
prob_df = results_df.loc[
    results_df[['CRPS', 'Coverage90', 'NLL']].notna().any(axis=1),
    [col for col in prob_cols if col in results_df.columns],
].sort_values(['horizon', 'CRPS'], na_position='last')
prob_df

## Best Statistical vs Fast ML

In [ ]:
comparison_path = PROJECT_ROOT / 'analysis' / 'stage1_model_search' / 'ml_vs_statistical_comparison.csv'
stat_ref = pd.read_csv(comparison_path)
stat_ref = stat_ref.loc[stat_ref['source_group'].eq('statistical_baseline')].copy()

# Use the same NRMSE convention as the statistical comparison table:
# NRMSE_zero = RMSE / RMSE_zero_baseline = sqrt(1 - OOS_R2).
ml_cmp = results_df.copy()
ml_cmp['nrmse_zero'] = np.sqrt(np.maximum(0.0, 1.0 - ml_cmp['OOS_R2']))

stat_result_paths = {
    10: PROJECT_ROOT / 'analysis' / 'target_10_model_experiments' / 'results' / 'latest' / 'experiment_results.parquet',
    20: PROJECT_ROOT / 'analysis' / 'target_10_model_experiments' / 'analysis' / 'target_10_model_experiments' / 'results' / 'target_20' / 'latest' / 'experiment_results.parquet',
    30: PROJECT_ROOT / 'analysis' / 'target_10_model_experiments' / 'results' / 'target_30' / 'latest' / 'experiment_results.parquet',
}

baseline_da_rows = []
for horizon, path in stat_result_paths.items():
    if not path.exists():
        continue
    stat_results = pd.read_parquet(path)
    baseline = stat_results.loc[stat_results['model_family'].eq('baseline_mean')]
    if baseline.empty:
        continue
    baseline_da_rows.append({
        'horizon': horizon,
        'baseline_da_025': float(baseline.iloc[0]['Direction_Accuracy_025']),
        'source_path': str(path.relative_to(PROJECT_ROOT)),
    })

baseline_da_df = pd.DataFrame(baseline_da_rows)
display(baseline_da_df)
baseline_da_by_horizon = dict(zip(baseline_da_df['horizon'], baseline_da_df['baseline_da_025']))
ml_cmp['baseline_da_025_used'] = ml_cmp['horizon'].map(baseline_da_by_horizon)
ml_cmp['da_lift_pct_points'] = (
    ml_cmp['Direction_Accuracy_0.25%'] - ml_cmp['baseline_da_025_used']
) * 100.0

rows = []
for horizon, group in ml_cmp.groupby('horizon'):
    stat_h = stat_ref.loc[stat_ref['horizon_min'].eq(horizon)]
    if stat_h.empty:
        continue

    stat_best_nrmse = stat_h.loc[stat_h['nrmse'].idxmin()]
    ml_best_nrmse = group.loc[group['nrmse_zero'].idxmin()]
    rows.append({
        'horizon': horizon,
        'metric': 'NRMSE_zero',
        'stat_model': stat_best_nrmse['model_label'],
        'stat_value': stat_best_nrmse['nrmse'],
        'ml_model': ml_best_nrmse['job_id'],
        'ml_value': ml_best_nrmse['nrmse_zero'],
        'ml_minus_stat': ml_best_nrmse['nrmse_zero'] - stat_best_nrmse['nrmse'],
        'winner': 'stat' if stat_best_nrmse['nrmse'] < ml_best_nrmse['nrmse_zero'] else 'ml',
    })

    stat_da = stat_h.dropna(subset=['da_lift_pct_points'])
    ml_da = group.dropna(subset=['da_lift_pct_points'])
    if not stat_da.empty and not ml_da.empty:
        stat_best_da = stat_da.loc[stat_da['da_lift_pct_points'].idxmax()]
        ml_best_da = ml_da.loc[ml_da['da_lift_pct_points'].idxmax()]
        rows.append({
            'horizon': horizon,
            'metric': 'DA_lift_pp',
            'stat_model': stat_best_da['model_label'],
            'stat_value': stat_best_da['da_lift_pct_points'],
            'ml_model': ml_best_da['job_id'],
            'ml_value': ml_best_da['da_lift_pct_points'],
            'ml_minus_stat': ml_best_da['da_lift_pct_points'] - stat_best_da['da_lift_pct_points'],
            'winner': 'stat' if stat_best_da['da_lift_pct_points'] > ml_best_da['da_lift_pct_points'] else 'ml',
        })

best_stat_vs_ml = pd.DataFrame(rows)
display(best_stat_vs_ml)

if not best_stat_vs_ml.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)

    nrmse_plot = best_stat_vs_ml.loc[best_stat_vs_ml['metric'].eq('NRMSE_zero')]
    axes[0].bar(nrmse_plot['horizon'].astype(str), nrmse_plot['ml_minus_stat'])
    axes[0].axhline(0, color='black', linewidth=1)
    axes[0].set_title('ML - Statistical best NRMSE')
    axes[0].set_xlabel('Horizon, minutes')
    axes[0].set_ylabel('Delta NRMSE; >0 means ML is worse')
    axes[0].grid(axis='y', alpha=0.25)

    da_plot = best_stat_vs_ml.loc[best_stat_vs_ml['metric'].eq('DA_lift_pp')]
    if not da_plot.empty:
        axes[1].bar(da_plot['horizon'].astype(str), da_plot['ml_minus_stat'])
    axes[1].axhline(0, color='black', linewidth=1)
    axes[1].set_title('ML - Statistical best DA lift')
    axes[1].set_xlabel('Horizon, minutes')
    axes[1].set_ylabel('Delta percentage points; <0 means ML is worse')
    axes[1].grid(axis='y', alpha=0.25)

    plt.show()


## Feature Importance Top 15

In [ ]:
importance_frames = []

for cfg in configs:
    horizon = int(cfg['horizon'])
    base = run_base(horizon, cfg['run_id'])
    for job_id in results_df.loc[results_df['horizon'].eq(horizon), 'job_id'].tolist():
        key = f'{base}/jobs/{job_id}/feature_importance.parquet'
        if not object_exists(key):
            continue
        frame = read_parquet(key)
        if frame.empty or not {'feature', 'importance'}.issubset(frame.columns):
            model_key = f'{base}/jobs/{job_id}/model.joblib'
            metadata_key = f'{base}/jobs/{job_id}/metadata.json'
            if not object_exists(model_key) or not object_exists(metadata_key):
                continue
            model = joblib.load(io.BytesIO(s3.get_object(Bucket=BUCKET, Key=model_key)['Body'].read()))
            metadata = read_json(metadata_key)
            features = metadata.get('features', [])
            if isinstance(model, dict):
                importances = []
                for estimator in model.values():
                    if hasattr(estimator, 'feature_importances_'):
                        importances.append(np.asarray(estimator.feature_importances_, dtype=float))
                if not importances:
                    continue
                importance = np.mean(np.vstack(importances), axis=0)
            elif hasattr(model, 'feature_importances_'):
                importance = np.asarray(model.feature_importances_, dtype=float)
            else:
                continue
            frame = pd.DataFrame({'feature': features, 'importance': importance})
        frame = frame.copy()
        frame['horizon'] = horizon
        frame['job_id'] = job_id
        frame['importance'] = pd.to_numeric(frame['importance'], errors='coerce').fillna(0.0)
        total = frame['importance'].abs().sum()
        frame['importance_norm'] = frame['importance'].abs() / total if total > 0 else 0.0
        importance_frames.append(frame)

importance_df = pd.concat(importance_frames, ignore_index=True) if importance_frames else pd.DataFrame()
importance_df[['horizon', 'job_id', 'feature', 'importance', 'importance_norm']].head()

In [ ]:
top15_by_model = (
    importance_df
    .sort_values(['horizon', 'job_id', 'importance_norm'], ascending=[True, True, False])
    .groupby(['horizon', 'job_id'], as_index=False)
    .head(15)
    .loc[:, ['horizon', 'job_id', 'feature', 'importance', 'importance_norm']]
    .reset_index(drop=True)
)

top15_by_model

In [ ]:
aggregate_top15 = (
    importance_df
    .groupby(['horizon', 'feature'], as_index=False)['importance_norm']
    .mean()
    .sort_values(['horizon', 'importance_norm'], ascending=[True, False])
    .groupby('horizon', as_index=False)
    .head(15)
    .reset_index(drop=True)
)

aggregate_top15

In [ ]:
if not aggregate_top15.empty:
    fig, axes = plt.subplots(len(HORIZONS), 1, figsize=(12, 4 * len(HORIZONS)), constrained_layout=True)
    if len(HORIZONS) == 1:
        axes = [axes]
    for ax, horizon in zip(axes, HORIZONS):
        plot_df = aggregate_top15.loc[aggregate_top15['horizon'].eq(horizon)].sort_values('importance_norm')
        ax.barh(plot_df['feature'], plot_df['importance_norm'])
        ax.set_title(f'Horizon {horizon}: aggregate top-15 normalized feature importance')
        ax.set_xlabel('Mean normalized importance across models with feature importance')
        ax.grid(axis='x', alpha=0.25)
    plt.show()

In [ ]:
# Change this to inspect a specific model's top-15 importance across horizons.
MODEL_FILTER = 'catboost_uncertainty_defaultish'

model_top15 = top15_by_model.loc[top15_by_model['job_id'].eq(MODEL_FILTER)].copy()
display(model_top15)

if not model_top15.empty:
    fig, axes = plt.subplots(len(HORIZONS), 1, figsize=(12, 4 * len(HORIZONS)), constrained_layout=True)
    if len(HORIZONS) == 1:
        axes = [axes]
    for ax, horizon in zip(axes, HORIZONS):
        plot_df = model_top15.loc[model_top15['horizon'].eq(horizon)].sort_values('importance_norm')
        ax.barh(plot_df['feature'], plot_df['importance_norm'])
        ax.set_title(f'{MODEL_FILTER}: horizon {horizon} top-15')
        ax.set_xlabel('Normalized importance')
        ax.grid(axis='x', alpha=0.25)
    plt.show()